# Heat Exchanger Network Synthesis
This notebook demonstrates the internal heat exchanger network design workflow on the converted OpenHENS Ye and Grossman four-stream problem. It uses the problem-owned `design` accessor for direct use with `PinchProblem`.

**Support notice:** `PinchProblem` and `PinchWorkspace` are public package-root workflows. Concrete owner modules used by this advanced notebook remain unsupported; `from OpenPinch.main import pinch_analysis_service` is the strict integration contract.


## Imports

In [ ]:
import pandas as pd
from OpenPinch import PinchProblem
from OpenPinch.presentation.network_grid.service import build_grid_diagram

## Ye and Grossman four-stream case
The stream and utility data come from the packaged `Four-stream-Yee-and-Grossmann-1990-1.json` conversion of the OpenHENS Ye and Grossman four-stream synthesis example. The packaged case keeps the original OpenHENS grid; this notebook narrows the grid to three approach temperatures, one derivative threshold, and one stage count so it can show the top three networks without sending Couenne through a broad teaching sweep.

In [ ]:
case_name = "Four-stream-Yee-and-Grossmann-1990-1.json"

teaching_grid = {
    "HENS_APPROACH_TEMPERATURES": [10.0, 14.0, 18.0],
    "HENS_DERIVATIVE_THRESHOLDS": [0.5],
    "HENS_STAGE_SELECTION": [3],
    "HENS_BEST_SOLUTIONS_TO_SAVE": 3,
    "HENS_MAX_PARALLEL": 1,
    "HENS_OUTPUT_FOLDER": "",
    "HENS_OUTPUT_FORMATS": [],
}

## Validate and prepare a problem
`PinchProblem` resolves the packaged sample-case name, validates the converted OpenHENS case input, and prepares the problem-owned `design` accessor used below.

In [ ]:
problem = PinchProblem(
    source=case_name,
    project_name="Four-stream converted OpenHENS example",
)
problem.update_options(teaching_grid)
validated = problem.validate()

stream_frame = pd.DataFrame(problem.master_zone.all_streams.to_dict())

configuration_frame = pd.DataFrame(
    [
        {
            "source": case_name,
            "project": problem.project_name,
            "streams": len(validated.streams),
            "utilities": len(validated.utilities or []),
            "approach_temperatures": validated.options["HENS_APPROACH_TEMPERATURES"],
            "derivative_thresholds": validated.options["HENS_DERIVATIVE_THRESHOLDS"],
            "stage_selection": validated.options["HENS_STAGE_SELECTION"],
        }
    ]
)

display(stream_frame)
configuration_frame

## Run the design workflow
The internal synthesis entry point is owned by the problem: `problem.design.heat_exchanger_network_synthesis()`. By default this call invokes the local solver-backed synthesis executor. Heat exchanger network configuration belongs in the case options loaded above; runtime options are reserved for target execution context such as `period_id`. Running this cell requires the `openpinch[synthesis]` Python extra and external solver binaries; `idaes get-extensions` installs Couenne and Ipopt into the IDAES extension directory, which OpenPinch will discover automatically.

In [ ]:
design = problem.design.heat_exchanger_network_synthesis()
top_ranked_network = design.ranked_networks[0]
network = design.network

pd.DataFrame(
    [
        {
            "run_id": design.run_id,
            "selected_rank": 1,
            "selected_method": design.method,
            "solver_name": design.solver_name,
            "solver_status": design.solver_status,
            "stage_count": design.stage_count,
            "ranked_network_count": len(design.ranked_networks),
            "exchanger_count": len(network.exchangers),
            "selected_network_is_rank_1": top_ranked_network.network == network,
            "cached_on_problem": problem.results.design == design,
        }
    ]
)

## Inspect the workflow manifest
The result carries serializable workflow metadata alongside the selected network.

In [ ]:
manifest = design.manifest

manifest_frame = pd.DataFrame(
    [
        {
            "run_id": manifest.run_id,
            "approach_temperatures": list(manifest.approach_temperatures),
            "derivative_thresholds": list(manifest.derivative_thresholds),
            "stage_selection": list(manifest.stage_selection),
            "task_ids": len(manifest.task_ids),
            "export_formats": list(manifest.export_formats),
        }
    ]
)

ranked_network_frame = pd.DataFrame(
    [
        {
            "method": outcome.task.method,
            "status": outcome.status,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
        }
        for outcome in design.ranked_networks
    ]
)

display(manifest_frame)
ranked_network_frame

## Top network grid designs
Successful network candidates are exposed as `design.ranked_networks` and ranked by objective value after duplicate exchanger-connection structures are removed. `design.network` is the top-ranked network by default, and `design.select_network(solution_rank=...)` changes the selected network. The presentation-owned `build_grid_diagram(...)` service renders the selected network. The Plotly grid diagrams below show the top three unique candidates from the accepted method family. Hover over exchanger markers to inspect match duty, area, and stream pairing.


In [ ]:
top_ranked_networks = design.get_n_best_networks(3)

ranked_network_summary = pd.DataFrame(
    [
        {
            "rank": rank,
            "method": outcome.task.method,
            "task_id": outcome.task.task_id,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
            "recovery_duty_kW": outcome.network.total_duty(
                kind="recovery", period_id=outcome.network.period_ids[0]
            ),
            "hot_utility_duty_kW": outcome.network.total_duty(
                kind="hot_utility", period_id=outcome.network.period_ids[0]
            ),
            "cold_utility_duty_kW": outcome.network.total_duty(
                kind="cold_utility", period_id=outcome.network.period_ids[0]
            ),
        }
        for rank, outcome in enumerate(top_ranked_networks, start=1)
    ]
)
display(ranked_network_summary)

selected_network_rows = [
    {
        "selected_rank": 1,
        "task_id": design.task_id,
        "objective_value": design.ranked_networks[0].objective_value,
        "exchanger_count": len(design.network.exchangers),
    }
]
if len(top_ranked_networks) >= 2:
    rank_2_design = design.model_copy(deep=True).select_network(solution_rank=2)
    selected_network_rows.append(
        {
            "selected_rank": 2,
            "task_id": rank_2_design.task_id,
            "objective_value": rank_2_design.ranked_networks[1].objective_value,
            "exchanger_count": len(rank_2_design.network.exchangers),
        }
    )

pd.DataFrame(selected_network_rows)

In [ ]:
grid_diagrams = []

for rank in range(1, min(3, len(top_ranked_networks)) + 1):
    design.select_network(solution_rank=rank)
    diagram = build_grid_diagram(design.network, period_id=design.network.period_ids[0])
    diagram.fig.update_layout(title_text=f"Rank {rank} heat exchanger network grid")
    grid_diagrams.append(diagram)
    display(diagram.fig)

design.select_network(solution_rank=1)
network = design.network

The returned diagram object wraps a Plotly figure. `diagram.save("network.png")` writes a static image through Kaleido, and `diagram.save("network.html")` writes an interactive Plotly document.


## Inspect the selected network
The selected `HeatExchangerNetwork` exposes ordered exchanger records plus labelled accessors for totals and stream-to-stream matches. Operating values live on ordered parent-owned period-state records; pass an explicit period identity whenever a network has more than one operating period. These runtime records remain internal.

In [ ]:
period_id = network.period_ids[0]
exchanger_frame = pd.DataFrame(
    [
        {
            "id": exchanger.exchanger_id,
            "kind": exchanger.kind.value,
            "source": exchanger.source_stream,
            "sink": exchanger.sink_stream,
            "stage": exchanger.stage,
            "period_id": state.period_id,
            "duty_kW": state.duty,
            "area": exchanger.area,
            "active": state.active,
            "match_allowed": exchanger.match_allowed,
            "source_outlet_K": state.source_outlet_temperature,
            "sink_outlet_K": state.sink_outlet_temperature,
        }
        for exchanger in network.exchangers
        for state in (exchanger.state(period_id),)
    ]
)

exchanger_frame

In [ ]:
first_recovery = next(
    exchanger for exchanger in network.exchangers if exchanger.kind == "recovery"
)
hot_utility_name = next(
    exchanger.source_stream
    for exchanger in network.exchangers
    if exchanger.kind == "hot_utility"
)
cold_utility_name = next(
    exchanger.sink_stream
    for exchanger in network.exchangers
    if exchanger.kind == "cold_utility"
)

pd.DataFrame(
    [
        {
            "metric": "total recovery duty",
            "value": network.total_duty(kind="recovery", period_id=period_id),
        },
        {
            "metric": "total hot utility duty",
            "value": network.total_duty(kind="hot_utility", period_id=period_id),
        },
        {
            "metric": "total cold utility duty",
            "value": network.total_duty(kind="cold_utility", period_id=period_id),
        },
        {
            "metric": f"{hot_utility_name} duty",
            "value": problem.design.network.utility(
                hot_utility_name, period_id=period_id
            ),
        },
        {
            "metric": f"{cold_utility_name} duty",
            "value": problem.design.network.utility(
                cold_utility_name, period_id=period_id
            ),
        },
        {
            "metric": "first recovery match duty",
            "value": network.labelled_value(
                "recovery_duty",
                source_stream=first_recovery.source_stream,
                sink_stream=first_recovery.sink_stream,
                stage=first_recovery.stage,
                period_id=period_id,
            ),
        },
    ]
)